In [1]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("SparkExample").getOrCreate()

print("Spark is running")

Spark is running


In [2]:
data = [
("Alice","HR",5000),
("Bob","HR",4000),
("Charlie","IT",6000),
("David","IT",5500),
("Eva","IT",7000),
("Frank","HR",4500)
]

columns = ["Name","Department","Salary"]
df = spark.createDataFrame(data,columns)

df.show()

+-------+----------+------+
|   Name|Department|Salary|
+-------+----------+------+
|  Alice|        HR|  5000|
|    Bob|        HR|  4000|
|Charlie|        IT|  6000|
|  David|        IT|  5500|
|    Eva|        IT|  7000|
|  Frank|        HR|  4500|
+-------+----------+------+



In [5]:
from pyspark.sql.window import Window   
from pyspark.sql.functions import row_number, rank, dense_rank

In [7]:
windowSpec = Window.partitionBy("Department").orderBy("Salary")

In [9]:
df.withColumn("RowNumber", row_number().over(windowSpec)).show()

+-------+----------+------+---------+
|   Name|Department|Salary|RowNumber|
+-------+----------+------+---------+
|    Bob|        HR|  4000|        1|
|  Frank|        HR|  4500|        2|
|  Alice|        HR|  5000|        3|
|  David|        IT|  5500|        1|
|Charlie|        IT|  6000|        2|
|    Eva|        IT|  7000|        3|
+-------+----------+------+---------+



In [11]:
df.withColumn("Rank", rank().over(windowSpec)).show()

+-------+----------+------+----+
|   Name|Department|Salary|Rank|
+-------+----------+------+----+
|    Bob|        HR|  4000|   1|
|  Frank|        HR|  4500|   2|
|  Alice|        HR|  5000|   3|
|  David|        IT|  5500|   1|
|Charlie|        IT|  6000|   2|
|    Eva|        IT|  7000|   3|
+-------+----------+------+----+



In [12]:
df.withColumn("DenseRank", dense_rank().over(windowSpec)).show()     

+-------+----------+------+---------+
|   Name|Department|Salary|DenseRank|
+-------+----------+------+---------+
|    Bob|        HR|  4000|        1|
|  Frank|        HR|  4500|        2|
|  Alice|        HR|  5000|        3|
|  David|        IT|  5500|        1|
|Charlie|        IT|  6000|        2|
|    Eva|        IT|  7000|        3|
+-------+----------+------+---------+



In [15]:
from pyspark.sql.functions import lag
df.withColumn("PreviousSalary", lag("Salary",1).over(windowSpec)).show()     #show the row value

+-------+----------+------+--------------+
|   Name|Department|Salary|PreviousSalary|
+-------+----------+------+--------------+
|    Bob|        HR|  4000|          NULL|
|  Frank|        HR|  4500|          4000|
|  Alice|        HR|  5000|          4500|
|  David|        IT|  5500|          NULL|
|Charlie|        IT|  6000|          5500|
|    Eva|        IT|  7000|          6000|
+-------+----------+------+--------------+



In [16]:
from pyspark.sql.functions import lead
df.withColumn("NextSalary", lead("Salary",1).over(windowSpec)).show()   #show the next row value

+-------+----------+------+----------+
|   Name|Department|Salary|NextSalary|
+-------+----------+------+----------+
|    Bob|        HR|  4000|      4500|
|  Frank|        HR|  4500|      5000|
|  Alice|        HR|  5000|      NULL|
|  David|        IT|  5500|      6000|
|Charlie|        IT|  6000|      7000|
|    Eva|        IT|  7000|      NULL|
+-------+----------+------+----------+

